In [1]:
from pathlib import Path

import cv2
import numpy as np

In [39]:
def load_ground_truth(path: Path) -> np.ndarray:
    """Load tab- or space-separated x, y, width, height boxes."""
    boxes = np.loadtxt(path, delimiter=None, dtype=np.float32)
    boxes = np.atleast_2d(boxes)

    if boxes.shape[1] != 4:
        raise ValueError(
            f"Expected four values per row, but found shape {boxes.shape}"
        )

    return boxes

In [40]:
def create_csrt_tracker():
    """
    Create a CSRT tracker while supporting different OpenCV Python layouts.
    """
    if hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()

    if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()

    raise RuntimeError(
        "CSRT is unavailable. Install opencv-contrib-python and make sure "
        "opencv-python is not installed in the same environment."
    )


In [41]:
def calculate_iou(box_a, box_b) -> float:
    """
    Calculate intersection over union.

    Both boxes use:
        x, y, width, height
    """
    ax, ay, aw, ah = map(float, box_a)
    bx, by, bw, bh = map(float, box_b)

    ax2 = ax + aw
    ay2 = ay + ah
    bx2 = bx + bw
    by2 = by + bh

    intersection_left = max(ax, bx)
    intersection_top = max(ay, by)
    intersection_right = min(ax2, bx2)
    intersection_bottom = min(ay2, by2)

    intersection_width = max(0.0, intersection_right - intersection_left)
    intersection_height = max(0.0, intersection_bottom - intersection_top)
    intersection_area = intersection_width * intersection_height

    area_a = max(0.0, aw) * max(0.0, ah)
    area_b = max(0.0, bw) * max(0.0, bh)

    union_area = area_a + area_b - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area

In [31]:
def draw_box(
    frame: np.ndarray,
    box,
    colour,
    label: str,
) -> None:
    x, y, width, height = box

    x = round(x)
    y = round(y)
    width = round(width)
    height = round(height)

    cv2.rectangle(
        frame,
        (x, y),
        (x + width, y + height),
        colour,
        2,
    )

    cv2.putText(
        frame,
        label,
        (x, max(20, y - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        colour,
        2,
    )

In [55]:
def inference(
    model: str,
    data_path: Path,
) -> None:
    # Preserve the validation folder structure to avoid duplicate filenames.
    # Example:
    # results/csrt/HSI-VIS-FalseColor/cat2.txt
    validation_type = data_path.parent.name
    sequence_name = data_path.name
    print(f"Processing: {validation_type}/{sequence_name}")

    output_directory = Path("results") / model / validation_type
    output_directory.mkdir(parents=True, exist_ok=True)

    output_video_path = output_directory / f"{sequence_name}.mp4"
    output_predictions_path = output_directory / f"{sequence_name}.txt"

    image_paths = sorted(data_path.glob("*.jpg"))

    if not image_paths:
        raise FileNotFoundError(f"No JPG images found in {data_path}")

    init_rect_path = data_path / "init_rect.txt"

    if not init_rect_path.is_file():
        raise FileNotFoundError(
            f"Initial bounding-box file not found: {init_rect_path}"
        )

    # Validation data should provide only the initial bounding box.
    initial_annotations = np.asarray(
        load_ground_truth(init_rect_path),
        dtype=np.float32,
    ).reshape(-1, 4)

    if len(initial_annotations) == 0:
        raise ValueError(f"No initial bounding box found in {init_rect_path}")

    initial_box = tuple(
        int(round(value))
        for value in initial_annotations[0]
    )

    first_frame = cv2.imread(str(image_paths[0]))

    if first_frame is None:
        raise RuntimeError(f"Could not read {image_paths[0]}")

    frame_height, frame_width = first_frame.shape[:2]

    x, y, width, height = initial_box

    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid initial box size: {initial_box}")

    if (
        x < 0
        or y < 0
        or x + width > frame_width
        or y + height > frame_height
    ):
        raise ValueError(
            f"Initial box {initial_box} is outside image dimensions "
            f"{frame_width}x{frame_height}"
        )
    print("Number of frames:", len(image_paths))
    print("First frame shape:", first_frame.shape)
    print("Initial box:", initial_box)

    tracker = create_csrt_tracker()
    tracker.init(first_frame, initial_box)

    video_writer = cv2.VideoWriter(
        str(output_video_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        20.0,
        (frame_width, frame_height),
    )

    if not video_writer.isOpened():
        raise RuntimeError(
            f"Could not create output video: {output_video_path}"
        )

    # Frame 1 uses the supplied initial bounding box.
    predictions = [initial_box]

    first_display = first_frame.copy()

    draw_box(
        first_display,
        initial_box,
        (0, 0, 255),
        "CSRT",
    )

    cv2.putText(
        first_display,
        "Frame: 1",
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
    )

    video_writer.write(first_display)

    try:
        for frame_index, image_path in enumerate(
            image_paths[1:],
            start=2,
        ):
            frame = cv2.imread(str(image_path))

            if frame is None:
                raise RuntimeError(f"Could not read {image_path}")

            tracking_success, predicted_box = tracker.update(frame)

            if tracking_success:
                predicted_box = tuple(
                    float(value)
                    for value in predicted_box
                )
            else:
                # Retain the previous prediction if tracking fails.
                predicted_box = predictions[-1]

            predictions.append(predicted_box)

            display_frame = frame.copy()

            draw_box(
                display_frame,
                predicted_box,
                (0, 0, 255),
                "CSRT",
            )

            cv2.putText(
                display_frame,
                f"Frame: {frame_index}",
                (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
            )

            if not tracking_success:
                cv2.putText(
                    display_frame,
                    "Tracking failure - previous box retained",
                    (15, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2,
                )

            video_writer.write(display_frame)

    finally:
        video_writer.release()
        cv2.destroyAllWindows()

    predictions_array = np.asarray(
        predictions,
        dtype=np.float32,
    )

    if len(predictions_array) != len(image_paths):
        raise RuntimeError(
            f"Generated {len(predictions_array)} predictions for "
            f"{len(image_paths)} frames."
        )

    np.savetxt(
        output_predictions_path,
        predictions_array,
        delimiter="\t",
        fmt="%.3f",
    )

    print(f"Predicted frames: {len(predictions_array)}")
    print(f"Predictions saved to: {output_predictions_path}")
    print(f"Visualisation saved to: {output_video_path}")

In [56]:
model = "csrt"
validation_data = Path("../data/validation/")

In [59]:
validations = [p for p in validation_data.iterdir() if p.is_dir()]

for validation in validations:
    datasets = [p for p in validation.iterdir() if p.is_dir()]
    for dataset in datasets:
        #print(datasets.name)
        inference(model, dataset)

Processing: HSI-VIS-FalseColor/motorcycle11
Number of frames: 251
First frame shape: (256, 512, 3)
Initial box: (278, 116, 10, 10)
Predicted frames: 251
Predictions saved to: results/csrt/HSI-VIS-FalseColor/motorcycle11.txt
Visualisation saved to: results/csrt/HSI-VIS-FalseColor/motorcycle11.mp4
Processing: HSI-VIS-FalseColor/pills7
Number of frames: 300
First frame shape: (272, 512, 3)
Initial box: (310, 160, 43, 44)
Predicted frames: 300
Predictions saved to: results/csrt/HSI-VIS-FalseColor/pills7.txt
Visualisation saved to: results/csrt/HSI-VIS-FalseColor/pills7.mp4
Processing: HSI-VIS-FalseColor/flower1
Number of frames: 239
First frame shape: (272, 512, 3)
Initial box: (358, 0, 72, 73)
Predicted frames: 239
Predictions saved to: results/csrt/HSI-VIS-FalseColor/flower1.txt
Visualisation saved to: results/csrt/HSI-VIS-FalseColor/flower1.mp4
Processing: HSI-VIS-FalseColor/earphone_case1
Number of frames: 442
First frame shape: (256, 512, 3)
Initial box: (126, 135, 99, 87)
Predicted f